# Evidence-Grounded Financial Anomaly Detector

This interactive demo runs **Prototype 2 only**: shared financial-data retrieval, deterministic anomaly detection, SEC Company Facts evidence retrieval, and one evidence-grounded explanation. Enter a ticker and its SEC CIK, then click **Run Prototype 2**. Live analysis requires `AI_API_KEY` in Colab Secrets; without a key, the interface displays the matching saved P2 result from `outputs1.json`. Never put an API key in notebook code.

## Start the demo

In Colab, add `AI_API_KEY` using the Secrets panel before running all cells if you want live LLM explanations. The notebook uses public Yahoo Finance and SEC EDGAR data. If live inference is unavailable, the saved P2 result is shown instead; a service error such as Gemini HTTP 503 is displayed as a failure.

In [ ]:
%pip install -q pandas numpy yfinance requests python-dotenv google-genai ipywidgets

from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/trishekle/ECE-pitch.git"
PROJECT_DIR = Path.cwd()
while PROJECT_DIR != PROJECT_DIR.parent and not (PROJECT_DIR / "src").is_dir():
    PROJECT_DIR = PROJECT_DIR.parent

if not (PROJECT_DIR / "src").is_dir():
    PROJECT_DIR = Path("/content/ECE-pitch")
    if not (PROJECT_DIR / "src").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(PROJECT_DIR)], check=True)

if not (PROJECT_DIR / "src").is_dir():
    raise FileNotFoundError(f"Project source directory not found: {PROJECT_DIR / 'src'}")
sys.path.insert(0, str(PROJECT_DIR))
print(f"Project ready: {PROJECT_DIR}")

In [ ]:
import json
import os
import time

import ipywidgets as widgets
import pandas as pd
from IPython.display import Markdown, clear_output, display

ticker_input = widgets.Text(
    value="AAPL",
    description="Ticker:",
    placeholder="e.g. AAPL",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="320px"),
)
cik_input = widgets.Text(
    value="320193",
    description="SEC CIK:",
    placeholder="e.g. 0000320193",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="320px"),
)
run_button = widgets.Button(
    description="Run Prototype 2",
    button_style="primary",
    icon="play",
    layout=widgets.Layout(width="220px", height="42px"),
)
result_area = widgets.Output(layout=widgets.Layout(border="1px solid #d0d7de", padding="12px"))
status = widgets.HTML(value="<i>Enter a ticker and SEC CIK, then run the evidence-grounded analysis.</i>")

def get_api_key():
    key = os.environ.get("AI_API_KEY")
    try:
        from google.colab import userdata
    except ImportError:
        return key
    try:
        return userdata.get("AI_API_KEY") or key
    except userdata.SecretNotFoundError:
        return key

def load_saved_p2(ticker):
    outputs_path = PROJECT_DIR / "outputs1.json"
    if not outputs_path.is_file():
        try:
            from google.colab import files
        except ImportError as exc:
            raise FileNotFoundError(f"Saved results not found: {outputs_path}") from exc
        print("Upload your saved outputs1.json file:")
        uploaded = files.upload()
        if "outputs1.json" not in uploaded:
            raise FileNotFoundError("Please upload the file named outputs1.json.")
        outputs_path.write_bytes(uploaded["outputs1.json"])

    saved_runs = json.loads(outputs_path.read_text(encoding="utf-8"))
    if not isinstance(saved_runs, list):
        raise ValueError("outputs1.json must contain a JSON list of saved runs.")
    matches = [
        row for row in saved_runs
        if row.get("prototype") == "P2"
        and row.get("output", {}).get("ticker", "").upper() == ticker
    ]
    if not matches:
        raise LookupError(f"No saved P2 result for {ticker}. Add AI_API_KEY in Colab Secrets to run a new live analysis.")
    return matches[-1]

def show_result(result, source, latency):
    anomaly_data = result.get("anomalies", {})
    anomaly_rows = anomaly_data.get("anomalies", []) if isinstance(anomaly_data, dict) else []
    if anomaly_rows:
        display(pd.DataFrame(anomaly_rows))
    else:
        display(pd.DataFrame([{"status": "No anomalies detected"}]))

    evidence = result.get("sec_evidence", {})
    evidence_names = ", ".join(evidence.keys()) if isinstance(evidence, dict) else "not available"
    display(Markdown(
        f"**Ticker:** {result.get('ticker', 'unknown')}  \n"
        f"**Result source:** {source}  \n"
        f"**Latency:** {latency if latency is not None else 'not recorded'} seconds  \n"
        f"**SEC fact concepts:** {evidence_names}"
    ))
    explanation = result.get("explanation")
    if explanation:
        display(Markdown("### Evidence-grounded explanation\n\n" + explanation))
    else:
        print("No explanation text was present in the result.")

def run_p2_from_gui(_):
    ticker = ticker_input.value.strip().upper()
    cik = cik_input.value.strip().lstrip("0")
    with result_area:
        clear_output(wait=True)
        if not ticker:
            status.value = "<b style='color:#b42318'>Enter a ticker symbol.</b>"
            return
        if not cik.isdigit():
            status.value = "<b style='color:#b42318'>Enter a numeric SEC CIK.</b>"
            return

        status.value = f"<i>Running P2 for {ticker}; retrieving public financial and SEC data...</i>"
        api_key = get_api_key()
        if api_key:
            os.environ["AI_API_KEY"] = api_key
            from google.genai.errors import ServerError
            from src.prototypes.p2_retrieval import run_p2

            started = time.perf_counter()
            try:
                result = run_p2(ticker, cik)
            except ServerError as exc:
                if exc.code == 503:
                    status.value = "<b style='color:#b42318'>P2 failed: Gemini returned HTTP 503 after retries. No successful result was produced.</b>"
                    print("Try again later, or run without a key to view a previously saved P2 result.")
                    return
                status.value = f"<b style='color:#b42318'>P2 failed: {type(exc).__name__}: {exc}</b>"
                return
            except Exception as exc:
                status.value = f"<b style='color:#b42318'>P2 failed: {type(exc).__name__}: {exc}</b>"
                return

            elapsed = round(time.perf_counter() - started, 3)
            status.value = f"<b style='color:#147d46'>P2 completed live for {ticker} in {elapsed:.2f} seconds.</b>"
            show_result(result, "live P2 run", elapsed)
            result_path = PROJECT_DIR / f"p2_result_{ticker}.json"
            result_path.write_text(json.dumps({"ticker": ticker, "cik": cik, "latency_seconds": elapsed, "output": result}, indent=2, default=str), encoding="utf-8")
            print(f"Saved result to {result_path.name}")
            return

        try:
            saved = load_saved_p2(ticker)
        except (FileNotFoundError, ValueError, LookupError) as exc:
            status.value = f"<b style='color:#b42318'>{exc}</b>"
            return
        result = saved["output"]
        status.value = f"<b style='color:#9a6700'>Showing saved P2 result for {ticker} (case {saved.get('case_id', 'unknown')}); this is not a new live run.</b>"
        show_result(result, "saved output", saved.get("latency_seconds"))

run_button.on_click(run_p2_from_gui)
display(widgets.VBox([
    widgets.HTML("<h3 style='margin-bottom:4px'>Evidence-grounded analysis</h3><p>Public market data + deterministic rules + SEC evidence + P2 explanation</p>"),
    ticker_input,
    cik_input,
    run_button,
    status,
    result_area,
], layout=widgets.Layout(gap="10px")))